# Loan pool tranching: real instruments behind a CLO waterfall

**Start here:** `02_pricing/instruments/structured_credit.ipynb` builds a CLO from asset rows;
`02_pricing/instruments/loans_and_credit_facilities.ipynb` prices term loans and revolvers on
their own. This notebook puts real instruments *inside* a deal: a floating bond, a callable bond,
a delayed-draw term loan and a stochastic revolver become the collateral of a two-tranche CLO whose
reserve account funds every draw and pays its interest to the equity tranche.

**Outcome:** price the deal deterministically, read the reserve path, then price it on stochastic
paths and decompose the **draw option cost** (the value the deal gives away by lending at a fixed
margin when spreads widen) by tranche.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`.

## Financial context and interpretation

A revolving facility inside a securitization is a written option: the borrower draws when its own
credit deteriorates and spreads widen, and the deal must fund the draw from its reserve account at
the *contractual* margin. Two mechanics matter:

- **Funding.** Draws are paid out of the reserve first, then out of that period's principal
  collections; whatever is left unfunded is capped and reported. Revolver repayments refill the
  reserve up to its target. The reserve earns interest that can go to the waterfall, a named
  tranche (here the equity) or stay in the reserve.
- **Adverse selection.** On the stochastic paths each revolver's spread follows a CIR process anchored
  to its hazard curve and its utilization target follows the spread (`spread_sensitivity`), so draws
  concentrate on the stress paths where names default.

The **draw option cost** values every funded draw as a forward loan at the contractual margin against
the path's fair spread. Each path is simulated twice on the same random numbers, once as is and
once with the draws accruing at the fair spread; a tranche's share is the difference in its present
value, and the shares sum to the deal cost on every path. A negative number is a cost to the deal.

In [ ]:
from datetime import date, timedelta
import json

import pandas as pd

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    HazardCurve,
    MarketContext,
    ScalarTimeSeries,
)
from finstack_quant.core.money import Money
from finstack_quant.valuations.instruments import (
    AssetPool,
    Bond,
    RevolvingCredit,
    StructuredCredit,
    TermLoan,
    Tranche,
    TrancheStructure,
    price_instrument,
)

AS_OF = date(2024, 1, 15)
MATURITY = date(2034, 1, 15)
USD = Currency("USD")

market = (
    MarketContext()
    .insert(DiscountCurve.flat("USD-OIS", AS_OF, 0.03))
    .insert(ForwardCurve("USD-SOFR-3M", 0.25, AS_OF, [(0.0, 0.04), (10.0, 0.04)], day_count="act_360"))
    # A rising hazard term structure: the borrower's fair spread widens over the life of the deal.
    .insert(HazardCurve("BORROWER-HZ", AS_OF, [(1.0, 0.04), (3.0, 0.10), (5.0, 0.14)], recovery_rate=0.4))
)
market.insert_series(
    ScalarTimeSeries("FIXING:USD-SOFR-3M", [(AS_OF - timedelta(days=d), 0.04) for d in range(25)])
)
print(market)

## Collateral: four real instruments

The bonds and the delayed-draw term loan are the canonical library examples. The revolver is built
with the typed builder: USD 30M committed, USD 10M drawn, SOFR + 250bp, a market-anchored spread
process on the borrower's hazard curve, 25% utilization volatility and a utilization target that
moves one-for-one with the relative spread change (`spread_sensitivity = 1.0`).

In [ ]:
revolver = (
    RevolvingCredit.builder()
    .id("RCF-BORROWER")
    .commitment_amount(Money(30_000_000.0, USD))
    .drawn_amount(Money(10_000_000.0, USD))
    .commitment_date(AS_OF)
    .maturity(date(2027, 1, 15))
    .base_rate_spec(RevolvingCredit.example().base_rate_spec)  # SOFR-3M + 250bp
    .day_count("act_360")
    .frequency("3M")
    .fees_flat(25.0, 10.0, 5.0)
    .draw_repay_spec(
        {
            "stochastic": {
                "utilization_process": {
                    "mean_reverting": {
                        "target_rate": 0.6,
                        "speed": 1.0,
                        "volatility": 0.25,
                        "spread_sensitivity": 1.0,
                    }
                },
                "num_paths": 32,
                "seed": 42,
                "antithetic": True,
                "use_sobol_qmc": False,
                "mc_config": {
                    "correlation_matrix": None,
                    "credit_spread_process": {
                        "market_anchored": {
                            "credit_curve_id": "BORROWER-HZ",
                            "kappa": 0.5,
                            "implied_vol": 0.4,
                            "tenor_years": None,
                        }
                    },
                    "interest_rate_process": None,
                    "util_credit_corr": 0.5,
                },
            }
        }
    )
    .discount_curve_id("USD-OIS")
    .credit_curve_id("BORROWER-HZ")
    .recovery_rate(0.4)
    .leq(0.5)
    .build()
)
assert revolver.is_stochastic

collateral = pd.DataFrame(
    [
        {"instrument": Bond.example_floating().id, "kind": "floating bond", "balance": Bond.example_floating().notional.amount},
        {"instrument": Bond.example_callable().id, "kind": "callable bond", "balance": Bond.example_callable().notional.amount},
        {"instrument": TermLoan.example_floating_with_ddtl().id, "kind": "delayed-draw term loan", "balance": TermLoan.example_floating_with_ddtl().notional_limit.amount},
        {"instrument": revolver.id, "kind": "stochastic revolver (drawn / committed)", "balance": f"{revolver.drawn_amount.amount:,.0f} / {revolver.commitment_amount.amount:,.0f}"},
    ]
)
print(collateral.to_string(index=False))

## The deal: instrument collateral, a reserve that funds draws and pays the equity

`AssetPool.with_instruments` accepts the typed instruments directly; `with_reserve` seeds USD 40M,
earns 3% and routes the interest to the equity tranche. The callable bond follows the deal-level
`first_call` policy. The tranches are a 90% senior note and a 10% equity note.

In [ ]:
pool = (
    AssetPool("LOAN-POOL", "clo", USD)
    .with_instruments(
        bonds=[Bond.example_floating(), Bond.example_callable()],
        term_loans=[TermLoan.example_floating_with_ddtl()],
        revolvers=[revolver],
        call_exercise="first_call",
    )
    .with_reserve(
        Money(40_000_000.0, USD),
        reserve_account_rate=0.03,
        reserve_target=Money(40_000_000.0, USD),
        reserve_interest_destination={"kind": "tranche", "tranche_id": "EQ"},
    )
)

equity = (
    Tranche.builder().id("EQ").attachment_point(0.0).detachment_point(10.0).seniority("equity")
    .original_balance(Money(5_200_000.0, USD)).coupon_fixed(0.0).maturity(MATURITY).build()
)
senior = (
    Tranche.builder().id("A").attachment_point(10.0).detachment_point(100.0).seniority("senior")
    .original_balance(Money(46_800_000.0, USD)).coupon_fixed(0.05).maturity(MATURITY).build()
)
deal = StructuredCredit.new_clo("LOAN-POOL-CLO", pool, TrancheStructure([equity, senior]), AS_OF, MATURITY, "USD-OIS")

# Pin the payment calendar and switch off the registry's behavioural CPR/CDR: the
# instruments' own hazard curves and schedules drive this deal.
envelope = json.loads(deal.to_json())
spec = envelope["instrument"]["spec"]
spec["payment_calendar_id"] = "nyse"
spec["prepayment_spec"] = {"cpr": 0.0, "curve": None}
spec["default_spec"] = {"cdr": 0.0, "curve": None}
deal = StructuredCredit.from_json(json.dumps(envelope))
print(deal)

## Deterministic valuation and the reserve path

The deterministic run uses each instrument's contractual schedule (the revolver contributes its
path-averaged expected schedule) and refuses a draw calendar the reserve cannot fund. The
diagnostics show the reserve being drawn by the delayed-draw loan and the revolver, then rebuilt
from revolver repayments, while its interest reaches the equity tranche every period.

In [ ]:
valuation = price_instrument(deal, market, AS_OF, "default")
print(f"deal PV: {valuation.value}")
assert valuation.value.amount > 0.0

diagnostics = deal.run_simulation_with_diagnostics(market, AS_OF)
reserve = diagnostics.to_dataframe()
print(f"draws funded from the reserve: {diagnostics.draws_from_reserve}")
print(f"revolver repayments recycled into the reserve: {diagnostics.reserve_replenished}")
print(f"unfunded draws: {diagnostics.unfunded_draws}")
print(reserve.head(8).to_string(index=False))

# Every draw is funded, the reserve never goes negative, and the equity receives the reserve interest.
assert diagnostics.unfunded_draws.amount == 0.0
assert (reserve["reserve_balance"] >= 0.0).all()
assert reserve["reserve_interest"].sum() > 0.0

## Stochastic valuation and the draw option cost by tranche

Each path draws per-name defaults, advances the revolver's spread and utilization and runs the full
waterfall. The result carries the expected draws, the fraction of paths on which a draw could not be
funded, and the draw option cost with its per-tranche shares and per-path distribution.

In [ ]:
result = deal.price_stochastic(market, AS_OF, num_paths=64)
print(f"deal PV: {result.npv}   expected loss: {result.expected_loss}")
print(f"expected collateral draws: {result.expected_collateral_draws}")
print(f"paths with an unfunded draw: {result.unfunded_draw_path_fraction:.1%}")
print(f"draw option cost: {result.draw_option_cost}")

tranches = result.to_dataframe()[["tranche_id", "seniority", "npv", "expected_loss", "draw_option_cost"]]
print(tranches.to_string(index=False))

distribution = result.draw_option_cost_dataframe()["draw_option_cost"]
print(distribution.describe().to_string())

# Widening spreads make the fixed-margin draws a cost to the deal, and the tranche shares
# sum to the deal cost.
assert result.draw_option_cost.amount < 0.0
assert abs(tranches["draw_option_cost"].sum() - result.draw_option_cost.amount) < 1e-6 * abs(result.draw_option_cost.amount)
assert len(distribution) == 64

## Reading the numbers

- The reserve path is the deal's liquidity story: it falls when the delayed-draw loan and the
  revolver draw, recovers from revolver repayments up to its target, and its interest is an
  equity-only cashflow that the waterfall never sees.
- The draw option cost is negative because the borrower's spread widens along the hazard curve
  while the facility keeps paying SOFR + 250bp. The equity tranche bears most of it: on a
  pass-through waterfall the residual interest is where the margin shortfall lands. Compare the
  standalone facility's `price_with_paths(...).draw_option_cost` to see the same economics without the
  waterfall.
- Re-run with `spread_sensitivity = 0.0` in the revolver's utilization process to switch off the
  adverse-selection channel and watch both the expected draws and the option cost shrink.